In [13]:
import sqlite3
import pandas as pd

ipo_clean = pd.read_csv('../data/processed/ipo_clean.csv')
price_history = pd.read_csv('../data/processed/price_history.csv')
nifty50 = pd.read_csv('../data/processed/nifty50_history.csv')

print(ipo_clean.shape)
print(price_history.shape)
print(nifty50.shape)

(397, 10)
(305199, 3)
(2090, 2)


In [14]:
#create a SQLite database
conn = sqlite3.connect('../data/processed/ipo_analysis.db')
ipo_clean.to_sql('ipo_clean',conn,if_exists ='replace',index=False)
price_history.to_sql('price_history',conn,if_exists ='replace',index=False)
nifty50.to_sql('nifty50',conn,if_exists ='replace',index=False)

2090

In [15]:
#calculating percentage of premium
ipo_clean['pop_pct'] = ((ipo_clean['listing_price'] - ipo_clean['issue_price']) / ipo_clean['issue_price']) * 100
ipo_clean.to_sql('ipo_clean',conn,if_exists = 'replace',index = False)
print(ipo_clean['pop_pct'].describe())
ipo_clean[['company_name','listing_date','nse_symbol','pop_pct']].head(10)

count    397.000000
mean      20.141939
std       35.776101
min     -100.000000
25%        0.000000
50%        9.090909
75%       32.027257
max      252.760736
Name: pop_pct, dtype: float64


,company_name,listing_date,nse_symbol,pop_pct
0,Gujarat Kidney & S...,2025-12-30,GKSL,5.921053
1,KSH International ...,2025-12-23,KSHINTL,-3.645833
2,ICICI Prudential A...,2025-12-19,ICICIAMC,20.378753
3,Nephrocare Health ...,2025-12-17,NEPHROPLUS,6.891304
4,Park Medi World Lt...,2025-12-17,PARKHOSPS,-3.950617
5,Wakefit Innovation...,2025-12-15,WAKEFIT,-100.000000
6,Corona Remedies Lt...,2025-12-15,CORONA,36.723164
7,Aequs Ltd,2025-12-10,AEQUS,12.903226
8,Meesho Ltd,2025-12-10,MEESHO,45.225225
9,Vidya Wires Ltd,2025-12-10,VIDYAWIRES,0.250000


In [16]:
# -100% pop seems like a source error
print(ipo_clean[ipo_clean['pop_pct'] < -50][['company_name', 'issue_price', 'listing_price', 'pop_pct']])

             company_name  issue_price  listing_price  pop_pct
5   Wakefit Innovation...          195            0.0   -100.0
28  Wework India Manag...          648            0.0   -100.0


In [17]:
#manual overide to correct the listing price
ipo_clean.loc[ipo_clean['company_name'].str.contains('Wakefit'),'listing_price'] = 195
ipo_clean.loc[ipo_clean['company_name'].str.contains('Wework'),'listing_price'] = 650

ipo_clean['pop_pct'] = ((ipo_clean['listing_price'] - ipo_clean['issue_price']) / ipo_clean['issue_price']) * 100
ipo_clean.to_sql('ipo_clean',conn,if_exists = 'replace',index = False)

397

In [18]:
print(ipo_clean['pop_pct'].describe())

count    397.000000
mean      20.646495
std       34.767695
min      -38.888889
25%        0.000000
50%        9.090909
75%       32.027257
max      252.760736
Name: pop_pct, dtype: float64
